# Business Model Canvas a partir do enriquecimento EaaS

Este notebook consome um export do `data-intelligence/eaas` (CSV, Parquet, JSON ou JSONL) ou consulta a API GraphQL em modo `QUERY_ONLY`. A saida tem uma linha por empresa, os nove blocos do Business Model Canvas, tags rastreaveis e indicadores de qualidade.

> O Canvas e uma hipotese para analise. Campos ausentes permanecem como `Nao identificado nos dados EaaS`; revise-os antes de uso comercial. Dados brutos e credenciais nao sao gravados no repositorio.

## Fluxo

1. carregar um export da tabela `company_enrichment`/`company_enrichment_current_state` ou consultar o GraphQL;
2. normalizar payload BigQuery e resposta GraphQL no mesmo contrato;
3. gerar um BMC-base conservador e tags com prefixos de origem;
4. opcionalmente completar hipoteses com um endpoint OpenAI-compatible;
5. validar cobertura e exportar CSV/Parquet em `analytics/data/` (diretorio ignorado pelo Git).

In [ ]:
from pathlib import Path
import json
import os
import sys
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import display

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPO_ROOT = next(path for path in candidates if (path / 'analytics' / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analytics.src.eaas_business_model_canvas import (
    CANVAS_BLOCKS,
    build_bmc_prompt,
    build_canvas_table,
    load_eaas_records,
    normalize_eaas_records,
    parse_bmc_response,
)

DATA_DIR = REPO_ROOT / 'analytics' / 'data'
SOURCE_MODE = os.getenv('EAAS_SOURCE_MODE', 'file').lower()  # file | api
EAAS_INPUT = Path(os.getenv('EAAS_ENRICHMENT_INPUT', DATA_DIR / 'eaas_enrichment.jsonl'))
IDENTIFIERS_INPUT = Path(os.getenv('EAAS_IDENTIFIERS_INPUT', DATA_DIR / 'eaas_identifiers.csv'))
EAAS_MODE = os.getenv('EAAS_MODE', 'QUERY_ONLY')
USE_DEMO_FALLBACK = os.getenv('EAAS_USE_DEMO', 'true').lower() == 'true'
USE_LLM = os.getenv('BMC_USE_LLM', 'false').lower() == 'true'
EXPORT_RESULTS = os.getenv('BMC_EXPORT_RESULTS', 'true').lower() == 'true'

assert SOURCE_MODE in {'file', 'api'}
assert EAAS_MODE in {'QUERY_ONLY', 'ENRICH_MISSING', 'FORCE_REFRESH'}
print({'source_mode': SOURCE_MODE, 'eaas_mode': EAAS_MODE, 'use_llm': USE_LLM})

## 1. Entrada EaaS

### Opcao A — arquivo (padrao)

Defina `EAAS_ENRICHMENT_INPUT` para um CSV, Parquet, JSON ou JSONL. O leitor aceita tanto colunas nativas com `payload` JSON quanto respostas GraphQL.

### Opcao B — API

Defina `EAAS_SOURCE_MODE=api`, crie `analytics/data/eaas_identifiers.csv` com colunas `cnpj` e/ou `domain` e forneca `EAAS_API_URL`, `EAAS_API_KEY` e `EAAS_USER_EMAIL` pelo ambiente. O padrao `QUERY_ONLY` evita enriquecimento ativo; altere o modo apenas de forma consciente.

In [ ]:
GRAPHQL_QUERY = r'''
query CanvasCompany($cnpj: String, $domain: String, $mode: EnrichmentMode!) {
  company(cnpj: $cnpj, domain: $domain, mode: $mode) {
    cnpj
    domain
    identity {
      name
      description
      receitaFederal { mainCnae }
    }
    digitalPresence {
      domainDetails { hasBlog technologies }
      socialMedia { linkedin facebook instagram tiktok pinterest twitterX youtube }
    }
    businessIntelligence {
      segment
      employees
      salesTeamSize
      competitors
      tags
      estimated { segment annualRevenue employees }
    }
    scores {
      porte { category }
      mopep { businessModel companyTags }
    }
    crmContext { relationshipRd relationshipTotvs }
    warnings
  }
}
'''

def fetch_eaas_company(cnpj=None, domain=None):
    api_url = os.environ.get('EAAS_API_URL')
    api_key = os.environ.get('EAAS_API_KEY')
    user_email = os.environ.get('EAAS_USER_EMAIL')
    if not all((api_url, api_key, user_email)):
        raise RuntimeError('Defina EAAS_API_URL, EAAS_API_KEY e EAAS_USER_EMAIL no ambiente.')
    body = json.dumps({
        'query': GRAPHQL_QUERY,
        'variables': {'cnpj': cnpj, 'domain': domain, 'mode': EAAS_MODE},
    }).encode('utf-8')
    request = Request(
        api_url,
        data=body,
        headers={
            'Content-Type': 'application/json',
            'X-API-Key': api_key,
            'X-User-Email': user_email,
        },
        method='POST',
    )
    with urlopen(request, timeout=90) as response:
        result = json.loads(response.read().decode('utf-8'))
    if result.get('errors'):
        raise RuntimeError(f"Erro GraphQL: {result['errors']}")
    return result

print('Cliente GraphQL pronto; nenhuma requisicao foi feita nesta celula.')

In [ ]:
DEMO_RECORDS = [{
    'cnpj': '00000000000100',
    'payload': json.dumps({
        'cnpj': '00000000000100',
        'domain': 'empresa-exemplo.invalid',
        'name': 'Empresa Exemplo',
        'description': 'Plataforma SaaS de automacao comercial para pequenas empresas.',
        'estimated_segment': 'software e cloud',
        'business_model': 'B2B SaaS',
        'company_size': {'company_size_category': 'SMB'},
        'employees': '50-100',
        'technologies': ['React', 'PostgreSQL'],
        'linkedin': 'https://linkedin.com/company/empresa-exemplo',
        'has_blog': True,
        'mopep': {'company_tags': ['SaaS', 'B2B']},
    }),
}]

if SOURCE_MODE == 'file' and EAAS_INPUT.exists():
    raw_records = load_eaas_records(EAAS_INPUT)
    input_label = str(EAAS_INPUT)
elif SOURCE_MODE == 'api':
    if not IDENTIFIERS_INPUT.exists():
        raise FileNotFoundError(f'Arquivo de identificadores nao encontrado: {IDENTIFIERS_INPUT}')
    identifiers = pd.read_csv(IDENTIFIERS_INPUT, dtype=str).fillna('')
    raw_records = [
        fetch_eaas_company(row.get('cnpj') or None, row.get('domain') or None)
        for row in identifiers.to_dict(orient='records')
        if row.get('cnpj') or row.get('domain')
    ]
    input_label = f'GraphQL ({EAAS_MODE})'
elif USE_DEMO_FALLBACK:
    raw_records = DEMO_RECORDS
    input_label = 'demonstracao local (nenhum dado real)'
else:
    raise FileNotFoundError(f'Entrada EaaS nao encontrada: {EAAS_INPUT}')

print(f'fonte={input_label} registros={len(raw_records)}')

In [ ]:
companies = normalize_eaas_records(raw_records)
normalized_df = pd.DataFrame(companies)

quality = pd.DataFrame({
    'metric': ['rows', 'identified_companies', 'with_description', 'with_segment', 'with_business_model'],
    'value': [
        len(normalized_df),
        normalized_df['company_id'].astype(bool).sum(),
        normalized_df['description'].notna().sum(),
        (normalized_df['segment'].notna() | normalized_df['estimated_segment'].notna()).sum(),
        normalized_df['business_model'].notna().sum(),
    ],
})
display(quality)
display(normalized_df[['company_id', 'name', 'domain', 'segment', 'estimated_segment', 'business_model']].head())

## 2. BMC-base e tags

O BMC-base reaproveita evidencia direta para proposta de valor e canais, e faz apenas inferencias simples sinalizadas como `hipotese`. `key_partners` e `cost_structure` permanecem vazios quando o EaaS nao possui base suficiente. As tags usam prefixos como `segment:`, `business-model:`, `eaas:` e `technology:` para preservar a origem semantica.

In [ ]:
canvas_df = pd.DataFrame(build_canvas_table(companies))
display(canvas_df.head())

canvas_view = canvas_df.set_index(['company_id', 'name'])[list(CANVAS_BLOCKS)].T
display(canvas_view)

In [ ]:
tags_long = (
    canvas_df[['company_id', 'name', 'tags']]
    .assign(tag=lambda frame: frame['tags'].str.split('|'))
    .explode('tag')
    .drop(columns='tags')
    .dropna(subset=['tag'])
)
tag_summary = (
    tags_long.groupby('tag', as_index=False)
    .agg(companies=('company_id', 'nunique'))
    .sort_values(['companies', 'tag'], ascending=[False, True])
)
display(tag_summary.head(30))

## 3. Complemento opcional por LLM

Ative somente com `BMC_USE_LLM=true`. Configure `BMC_LLM_BASE_URL`, `BMC_LLM_MODEL` e, se necessario, `BMC_LLM_API_KEY`. O prompt envia apenas campos corporativos permitidos pelo normalizador; contatos e outros dados pessoais nao sao incluidos. Verifique a politica de dados antes de apontar para um provedor externo.

In [ ]:
def chat_completions_url(base_url):
    base = base_url.rstrip('/')
    return f'{base}/chat/completions' if base.endswith('/v1') else f'{base}/v1/chat/completions'

def call_bmc_llm(company):
    base_url = os.environ.get('BMC_LLM_BASE_URL')
    model = os.environ.get('BMC_LLM_MODEL')
    api_key = os.environ.get('BMC_LLM_API_KEY')
    if not base_url or not model:
        raise RuntimeError('Defina BMC_LLM_BASE_URL e BMC_LLM_MODEL.')
    headers = {'Content-Type': 'application/json'}
    if api_key:
        headers['Authorization'] = f'Bearer {api_key}'
    body = json.dumps({
        'model': model,
        'messages': [
            {'role': 'system', 'content': 'Voce e um analista de modelos de negocio rigoroso e conservador.'},
            {'role': 'user', 'content': build_bmc_prompt(company)},
        ],
        'temperature': 0.1,
        'response_format': {'type': 'json_object'},
    }).encode('utf-8')
    request = Request(chat_completions_url(base_url), data=body, headers=headers, method='POST')
    with urlopen(request, timeout=180) as response:
        result = json.loads(response.read().decode('utf-8'))
    content = result['choices'][0]['message']['content']
    return parse_bmc_response(content)

if USE_LLM:
    llm_results = [call_bmc_llm(company) for company in companies]
    for index, result in enumerate(llm_results):
        for block in CANVAS_BLOCKS:
            canvas_df.at[index, block] = result[block]
        canvas_df.at[index, 'bmc_source'] = 'llm-reviewed-hypothesis'
        canvas_df.at[index, 'llm_confidence'] = result.get('confidence')
        canvas_df.at[index, 'bmc_assumptions'] = json.dumps(result.get('assumptions', []), ensure_ascii=False)
        canvas_df.at[index, 'llm_tags'] = '|'.join(result.get('tags', []))
    print(f'BMC complementado por LLM para {len(llm_results)} empresa(s).')
else:
    print('Etapa LLM desativada; mantendo baseline auditavel do EaaS.')

In [ ]:
unknown_by_block = pd.DataFrame({
    'block': CANVAS_BLOCKS,
    'unknown_rows': [canvas_df[block].str.startswith('Nao identificado').sum() for block in CANVAS_BLOCKS],
})
unknown_by_block['coverage_pct'] = (
    100 * (1 - unknown_by_block['unknown_rows'] / max(len(canvas_df), 1))
).round(1)

assert canvas_df['company_id'].fillna('').astype(bool).all(), 'Ha empresa sem identificador.'
assert not canvas_df.duplicated('company_id').any(), 'Ha company_id duplicado; consolide a versao mais recente.'
assert set(CANVAS_BLOCKS).issubset(canvas_df.columns)
display(unknown_by_block)
display(canvas_df[['company_id', 'bmc_source', 'evidence_coverage', 'needs_review']])

In [ ]:
OUTPUT_CSV = DATA_DIR / 'eaas_business_model_canvas.csv'
OUTPUT_PARQUET = DATA_DIR / 'eaas_business_model_canvas.parquet'
OUTPUT_TAGS = DATA_DIR / 'eaas_business_model_canvas_tags.csv'

if EXPORT_RESULTS:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    canvas_df.to_csv(OUTPUT_CSV, index=False)
    canvas_df.to_parquet(OUTPUT_PARQUET, index=False)
    tags_long.to_csv(OUTPUT_TAGS, index=False)
    print(f'csv={OUTPUT_CSV}')
    print(f'parquet={OUTPUT_PARQUET}')
    print(f'tags={OUTPUT_TAGS}')
else:
    print('Exportacao desativada por BMC_EXPORT_RESULTS=false.')

## Criterios para uso

- filtre ou revise toda linha com `needs_review=True`;
- use `evidence_coverage` para priorizar empresas com melhor base;
- mantenha as tags prefixadas para nao misturar dado EaaS com hipotese do Canvas;
- valide principalmente `customer_segments`, `key_partners`, `revenue_streams` e `cost_structure`, que raramente sao observaveis apenas pelo enriquecimento corporativo.